In [1]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

print("\n--- NGÀY 69: TỪ ĐIỂN SỐ HỌC (TOKENIZATION & PADDING) ---")

#1.Giả lập đánh giá của KH
danh_gia_cua_kh = [
    "Sản phẩm này rất tốt",
    "Rất tệ, tôi không thích sản phẩm này",
    "Tốt",
    "Giao hàng nhanh, sản phẩm đẹp xuất sắc"
]

#2.Khởi tạo người lập từ điển (Tokenizer)
#num_row=100 chỉ lưu lại 100 từ phổ biến nhất
#ovv_token="LẠ" thay thế các từ không có trong từ điển là LẠ
may_dich = Tokenizer(num_words=100, oov_token="<LẠ>")
#cho máy đọc qua dữ liệu, để tạo ra từ điển từ nào xuất hiện nhiều nhất
may_dich.fit_on_texts(danh_gia_cua_kh)

#In cuốn từ điển
tu_dien =may_dich.word_index
print(f"Từ điển AI vừa lập được (Tổng cộng {len(tu_dien)} từ):")
print(tu_dien)

#3.Chuyển chữ thành chuỗi số
chuoi_so = may_dich.texts_to_sequences(danh_gia_cua_kh)
print("\n Các câu văn sau khi chuyển thành chuỗi số")

for i in range(len(danh_gia_cua_kh)):
    print(f"[{danh_gia_cua_kh[i]}] ---> {chuoi_so[i]}")
    

#4.Kéo dãn/cắt gọt cho bằng nhau (Padding)
du_lieu_chuan = pad_sequences(chuoi_so, maxlen=7, padding='pre')

print("\n--- MA TRẬN CUỐI CÙNG ĐỂ ĐƯA VÀO MẠNG NƠ-RON ---")
print(du_lieu_chuan)


--- NGÀY 69: TỪ ĐIỂN SỐ HỌC (TOKENIZATION & PADDING) ---
Từ điển AI vừa lập được (Tổng cộng 16 từ):
{'<LẠ>': 1, 'sản': 2, 'phẩm': 3, 'này': 4, 'rất': 5, 'tốt': 6, 'tệ': 7, 'tôi': 8, 'không': 9, 'thích': 10, 'giao': 11, 'hàng': 12, 'nhanh': 13, 'đẹp': 14, 'xuất': 15, 'sắc': 16}

 Các câu văn sau khi chuyển thành chuỗi số
[Sản phẩm này rất tốt] ---> [2, 3, 4, 5, 6]
[Rất tệ, tôi không thích sản phẩm này] ---> [5, 7, 8, 9, 10, 2, 3, 4]
[Tốt] ---> [6]
[Giao hàng nhanh, sản phẩm đẹp xuất sắc] ---> [11, 12, 13, 2, 3, 14, 15, 16]

--- MA TRẬN CUỐI CÙNG ĐỂ ĐƯA VÀO MẠNG NƠ-RON ---
[[ 0  0  2  3  4  5  6]
 [ 7  8  9 10  2  3  4]
 [ 0  0  0  0  0  0  6]
 [12 13  2  3 14 15 16]]


In [ ]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, GlobalAveragePooling1D

print("\n--- NGÀY 70: HUẤN LUYỆN AI PHÂN TÍCH CẢM XÚC (SENTIMENT ANALYSIS) ---")

# 1. Dán nhãn cho dữ liệu (1: Khen / Tích cực, 0: Chê / Tiêu cực)
# Câu 1: "Sản phẩm này rất tốt" -> 1
# Câu 2: "Rất tệ..." -> 0
# Câu 3: "Tốt" -> 1
# Câu 4: "Giao hàng nhanh..." -> 1
nhan_cam_xuc = np.array([1,0,1,1])#Đây là mảng tượng chưng cho phần lời khen của khách hàng, ví dụ câu 1 là lời khen nên được gán nhãn 1

#2. Xây dựng mô hình AI NLP (Nautural Language Processing)
model_nlp = Sequential()

#Thêm lớp nhúng (Embedding) để chuyển đổi số thành vector đặc trưng
#input_dim=100 số lượng từ trong từ điển mà ta đã định nghĩa
#output_din=16 ép mỗi từ thành vector 16 chiều
#input_length=7 độ dài của mỗi câu sau khi đã được padding
model_nlp.add(Embedding(input_dim = 100, output_dim=16, input_length=7))

#Lớp globalaverangepooling1d: lấy trung bình ý nghĩa của 7 từ trong câu
#Giúp Ai tóm tắt ý chính của câu
model_nlp.add(GlobalAveragePooling1D())

#Lớp ẩn
model_nlp.add(Dense(16,activation='relu'))

#lớp đầu ra 1 nơ-ron (vì kết quả chỉ cần khen/chê)
model_nlp.add(Dense(1,activation='sigmoid'))

#3. Biên dịch mô hình
model_nlp.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

#4.Huấn luyện mô hình
print("Bắt đầu huấn luyện khả năng đọc hiểu...")
lich_su_hoc_nlp = model_nlp.fit(
    du_lieu_chuan,
    nhan_cam_xuc,
    epochs=50,
    verbose=1
)
print("Huấn luyện xong!")

#5. Dự đoán cảm xúc của câu mới
cau_moi = ["Sản phẩm này tệ quá"]
print(f"\nKhách hàng mới bình luận: '{cau_moi[0]}'")

#Tiền xử lí dữ liệu
#text_to_sequences(cau_moi) cầm quấn từ điển đã học lên để dịch câu mới thành số
so_hoa_cau_moi = may_dich.texts_to_sequences(cau_moi)
cau_moi_chuan = pad_sequences(so_hoa_cau_moi, maxlen=7, padding='pre')

#Cho AI dự đoán
du_doan = model_nlp.predict(cau_moi_chuan)

if du_doan[0][0] > 0.5:
    print(f"=> AI phán: Đây là lời KHEN (Tích cực) - Độ tự tin: {du_doan[0][0]*100:.2f}%")
else:
    print(f"=> AI phán: Đây là lời CHÊ (Tiêu cực) - Độ tự tin: {(1 - du_doan[0][0])*100:.2f}%")

print(f"AI đã thực sự học được: {len(lich_su_hoc_nlp.history['loss'])} vòng")

In [ ]:
from tensorflow.keras.layers import LSTM

print("\n--- NGÀY 71: DẠY AI ĐỌC THEO NGỮ CẢNH (LSTM) ---")

model_lstm = Sequential()

#1.Thêm lớp nhúng (Embedding) để chuyển đổi số thành vector đặc trưng
model_lstm.add(Embedding(input_dim=100, output_dim=16, input_length=7))

#2.Lớp trí nhớ LSTM (thay cho lớp GlobaAveragePool1D) cũng là lớp ẩn
#LSTM(16) nhận vào vector 16 chiều từ lớp nhúng, và học cách nhớ/ngắt quãng thông tin trong câu
model_lstm.add(LSTM(16))

#3.Lớp đầu ra
model_lstm.add(Dense(1,activation='sigmoid'))

# Biên dịch mô hình
model_lstm.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Huấn luyện (Ép nó học 100 vòng cho nhớ thật kỹ)
print("Bắt đầu huấn luyện não bộ LSTM...")
model_lstm.fit(du_lieu_chuan, nhan_cam_xuc, epochs=100, verbose=0)
print("Huấn luyện xong!")

# 4. CHO AI THI LẠI VỚI CÂU VĂN CŨ
cau_moi = ["Sản phẩm này tệ quá"]
print(f"\nKhách hàng mới bình luận: '{cau_moi[0]}'")

so_hoa_cau_moi = may_dich.texts_to_sequences(cau_moi)
cau_moi_chuan = pad_sequences(so_hoa_cau_moi, maxlen=7, padding='pre')

du_doan_lstm = model_lstm.predict(cau_moi_chuan)

if du_doan_lstm[0][0] > 0.5:
    print(f"=> AI (LSTM) phán: Đây là lời KHEN (Tích cực) - Độ tự tin: {du_doan_lstm[0][0]*100:.2f}%")
else:
    print(f"=> AI (LSTM) phán: Đây là lời CHÊ (Tiêu cực) - Độ tự tin: {(1 - du_doan_lstm[0][0])*100:.2f}%")